![mental_health_banner_1774025040503.jpg](../mental_health_banner_1774025040503.jpg "mental_health_banner_1774025040503.jpg")

### Ingestion de Datos

Ingesta de Datos Patients Information

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "health_db_catalog")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "saafadproject")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/patients_information.csv"

In [0]:
patients_schema = StructType([
    StructField("person_id", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("occupation", StringType(), True),
    StructField("state", IntegerType(), True)    
])

In [0]:
df_patients = (
    spark
    .read
    .option("header", True)
    .schema(patients_schema)
    .csv(ruta)
)

In [0]:
df_patients_final = (df_patients.select(col("person_id"),
                                        col("gender"),
                                        col("age"),
                                        col("occupation"),
                                        col("state")
                                        ).withColumn("ingestion_date", current_timestamp())
)

In [0]:
df_patients_final.coalesce(1).write.mode("overwrite").option("mergeSchema", "true").insertInto(f"{catalogo}.{esquema}.patients_information")